# 02 数据清洗与质量检查

输入：`data/sample/*.csv`。输出：`reports/notebook_outputs/data_quality_summary.csv`。检查主键、缺失值、重复值和时间范围。

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(p for p in ROOT.parents if (p / 'src').exists())
sys.path.insert(0, str(ROOT / 'src'))
from ashare_factor_research.data.data_loader import LocalDataLoader
data = LocalDataLoader(ROOT / 'data' / 'sample').load_all()
rows = []
for name, df in data.items():
    key_cols = [c for c in ['trade_date', 'ts_code'] if c in df.columns]
    date_col = 'trade_date' if 'trade_date' in df.columns else 'publish_date' if 'publish_date' in df.columns else None
    rows.append({'table': name, 'rows': len(df), 'columns': len(df.columns), 'missing_cells': int(df.isna().sum().sum()), 'duplicate_key_rows': int(df.duplicated(key_cols).sum()) if key_cols else 0, 'start_date': df[date_col].min() if date_col else pd.NaT, 'end_date': df[date_col].max() if date_col else pd.NaT})
quality = pd.DataFrame(rows)
out_dir = ROOT / 'reports' / 'notebook_outputs'
out_dir.mkdir(parents=True, exist_ok=True)
quality.to_csv(out_dir / 'data_quality_summary.csv', index=False)
print(quality.to_string(index=False))